# 01 — AKShare 补充源端到端验证

本 notebook 验证 chinaStock 中 AKShare 集成的端到端可用性：

1. AKShare 安装与版本
2. 龙虎榜集成（异动归因）
3. 涨停池 + 情绪计（情绪温度）
4. 板块/概念（主线确认）
5. 缓存文件检查

**前置条件**：项目根目录执行 `pip install -r requirements.txt`

**注意**：本 notebook 会真实调用 AKShare 接口，需联网。失败单元会捕获并打 warning，不影响其他单元运行。

In [ ]:
import sys
sys.path.insert(0, ".")  # 从项目根目录导入

import akshare as ak
print(f"AKShare version: {ak.__version__}")
import pandas as pd
print(f"pandas version: {pd.__version__}")

## 2. 龙虎榜集成

调 `get_lhb()` 拉指定票近期龙虎榜。结果应包含 `symbol, name, net_buy_amount` 等 snake_case 列，`symbol` 列以 `SH`/`SZ`/`BJ` 前缀开头。

In [ ]:
from src.integrations.lhb import get_lhb, explain_anomaly
from datetime import datetime, timedelta

# 取最近一个交易日（向前 30 天兜底）
candidate_date = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"查询日期: {candidate_date}")

df_lhb = get_lhb("SH600519", candidate_date)
print(f"返回行数: {len(df_lhb)}")
print(f"列名: {df_lhb.columns.tolist()}")
if not df_lhb.empty:
    display(df_lhb.head())
else:
    print("(空结果 — 该日该票未上榜；可换其他日期/股票重试)")

## 3. 涨停池 + 情绪计

调 `market_sentiment_score()` 拿到三维情绪分：涨停数 / 最高连板 / 炸板率。

In [ ]:
from src.integrations.limit_up import (
    get_limit_up_pool, get_limit_up_streak, market_sentiment_score,
)

score = market_sentiment_score(candidate_date)
print("市场情绪计:")
for k, v in score.items():
    print(f"  {k}: {v}")

pool = get_limit_up_pool(candidate_date)
print(f"\n涨停池大小: {len(pool)}")
if not pool.empty:
    print(f"列名: {pool.columns.tolist()}")
    display(pool.head())

streak = get_limit_up_streak(candidate_date, min_boards=2)
print(f"\n2 连板及以上: {len(streak)} 只")
if not streak.empty:
    display(streak[["symbol", "name", "consecutive_boards"]].head(10))

## 4. 板块/概念

调 `get_sector_performance()` 拉概念板块日 K（与个股 K 线同 schema）。

In [ ]:
from src.integrations.sectors import (
    get_sector_constituents, get_sector_performance,
)

SECTOR = "机器人"
start = (datetime.today() - timedelta(days=60)).strftime("%Y-%m-%d")
end = candidate_date
print(f"板块: {SECTOR}, 区间: {start} ~ {end}")

members = get_sector_constituents(SECTOR)
print(f"\n成分股数: {len(members)}")
if not members.empty:
    print(f"列名: {members.columns.tolist()}")
    display(members.head(10))

perf = get_sector_performance(SECTOR, start, end)
print(f"\n板块 K 线行数: {len(perf)}")
if not perf.empty:
    print(f"列名: {perf.columns.tolist()}")
    display(perf.head())
    display(perf.tail())

## 5. 缓存文件检查

以上调用应已生成 parquet 缓存到 `data/cache/`。

In [ ]:
from pathlib import Path
from src.data_layer.cache import CACHE_DIR

if not CACHE_DIR.exists():
    print(f"缓存目录不存在: {CACHE_DIR}")
else:
    files = sorted(CACHE_DIR.glob("*.parquet"))
    print(f"缓存目录: {CACHE_DIR}")
    print(f"文件数: {len(files)}\n")
    for f in files:
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name}  ({size_kb:.1f} KB)")

    if files:
        print("\n✓ 缓存生效，第二次相同参数调用将不会触发网络请求")